**Question:** Does blending the sequence model (GRU, Exp C3) with a tabular model (XGBoost B' or TabPFN v2, both on the same 37 features) beat the GRU on its own, including the 5-seed GRU ensemble?

**Assumptions:** Same test matches as notebooks 14 and 15 (walk-forward folds 1-4, 14,428 matches pooled). All component models are isotonic-calibrated on the calibration slice of their fold. Blend weights are fixed in advance (0.5 / 0.5, and 1/3 each for the 3-way blend); nothing is fitted on test data. C3ens is the mean of 5 GRU seeds; C3, XGBoost and TabPFN v2 are single-seed.

**Method:** XGBoost B' is re-trained walk-forward (`scripts/run_walkforward_xgb_bprime.py`) with the hyper-parameters of notebook 09. Paired bootstrap by match (2,000 resamples, seed 42, same indices for every comparison), pooled over 4 folds; per-fold Brier and number of folds won; correlation of residuals; improvement over ELO.

**Result:** A GRU + tabular blend is significantly better than the GRU alone and than the 5-seed GRU ensemble (paired dBrier CI below 0). The XGBoost and TabPFN v2 partners give statistically indistinguishable blends, so the cheaper XGBoost partner is enough. The TabPFN-3.5 blend is shown for research only.

# 16 -- GRU + tabular blends (XGBoost B' and TabPFN v2)

**Why blend.** The GRU (C3) reads the match history as a sequence; XGBoost and TabPFN read a table of 37 hand-built features (ELO difference, form, rest, league context, trajectory). Their errors are correlated but not identical, so an average can be better than either part. Residual correlations are reported in Step 4.

**Setup.** XGBoost B' (37 features, walk-forward on the same folds) is trained locally by `scripts/run_walkforward_xgb_bprime.py` into `<workdir>/exp_xgb`. TabPFN v2 (10k context, seed 42) comes from notebook 14 (`exp_d`), TabPFN-3.5 (full context) from notebook 15 (`exp_d35`). Blends are plain probability averages with fixed weights.

**Licence note.** TabPFN-3.5 is used under the TabPFN-3.5 Non-Commercial License for research evaluation only; the blend containing it is a research result and never goes to production.

**Statistics.** Paired bootstrap by match (both perspective rows together). Because all models share the same test matches, absolute Brier CIs overlap; the paired dBrier CI is the relevant comparison. Negative dBrier = first model is better. No predictions or weights are stored in this notebook.

## Step 0 -- Imports and data

In [ ]:
import sys
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '../scripts')
from exp_d_analysis import (load_folds, add_d35, add_xgb, add_blends, pool_folds, met, brier_ci,
                            paired_dbrier, boot_idx, summary_table, pair_dev, symmetrize)

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)

WORKDIR = '..'
# XGBoost B' predictions are produced once by the training script (skipped if already present)
if not (Path(WORKDIR) / 'exp_xgb' / 'test_predictions_xgb_bprime_fold4.parquet').exists():
    subprocess.run([sys.executable, '../scripts/run_walkforward_xgb_bprime.py', WORKDIR], check=True)

V2 = 'p_D-Bprime__last10k__s42'
B = add_xgb(add_d35(load_folds(WORKDIR), WORKDIR), WORKDIR)
BL = add_blends(B)
LAB = {'p_ELO': 'ELO', 'p_xgb': "XGBoost B'", 'p_C3': 'C3 (calibrated)', 'p_C3raw': 'C3 raw (production)', 'p_C3ens_raw': 'C3ens raw (5 seeds)', 'p_C3ens': 'C3ens (5 seeds, calibrated)', V2: 'TabPFN v2 (10k)',
       'p_d35_full': 'TabPFN-3.5 (full, research-only)'}
LAB.update(BL)
pool = pool_folds(B)
yp = pool.Label.values
boots = boot_idx(pool.SusretId.values)
print('folds:', sorted(B), '| pooled matches:', pool.SusretId.nunique(), '| rows:', len(pool))

## Step 1 -- XGBoost B' walk-forward (Brier by fold)

The training script is the single source of these predictions; hyper-parameters and the chronological train / early-stopping / calibration split are the same as in notebook 09.

In [ ]:
rows = [{'fold': k, 'n_matches': B[k].SusretId.nunique(),
         'ELO': np.mean((B[k].p_ELO - B[k].Label) ** 2),
         "XGBoost B'": np.mean((B[k].p_xgb - B[k].Label) ** 2),
         'C3': np.mean((B[k].p_C3 - B[k].Label) ** 2)} for k in sorted(B)]
XT = pd.DataFrame(rows)
print(XT.round(5).to_string(index=False))
b, lo, hi = brier_ci(pool.p_xgb.values, yp, boots)
print(f"\npooled XGBoost B' Brier {b:.5f} [{lo:.5f}, {hi:.5f}]")

## Step 2 -- TabPFN vs XGBoost on the same 37 features

Same inputs, same folds, same calibration: this isolates the model class. Negative = first model better.

In [ ]:
print('Brier by fold')
for n, c in (("XGBoost B'", 'p_xgb'), ('TabPFN v2 (10k)', V2), ('TabPFN-3.5 (full)', 'p_d35_full')):
    print(f'  {n:20s}', [round(float(np.mean((B[k][c] - B[k].Label) ** 2)), 5) for k in sorted(B)])
print()
for a, b_ in ((V2, 'p_xgb'), ('p_d35_full', 'p_xgb'), ('p_d35_full', V2), ('p_xgb', 'p_C3'), ('p_xgb', 'p_C3ens')):
    d, lo, hi = paired_dbrier(pool[a].values, pool[b_].values, yp, boots)
    w = sum(np.mean((B[k][a] - B[k].Label) ** 2) < np.mean((B[k][b_] - B[k].Label) ** 2) for k in B)
    print(f'{LAB[a]:34s} vs {LAB[b_]:18s} dBrier {d:+.5f} [{lo:+.5f}, {hi:+.5f}]   folds better: {w}/4')

On identical features TabPFN v2 beats XGBoost by a small but statistically clear margin; TabPFN-3.5 beats it by more. Both tabular models are clearly worse than the GRU on its own, so a tabular model is a **partner** for the GRU, not a replacement.

## Step 3 -- Blends with fixed weights

Weights are fixed before looking at the test folds (equal weights; 1/3 each in the 3-way blend). No weight is fitted on test data, so the blend CIs carry no selection bias from weight tuning. The blend with TabPFN-3.5 is included for research reference only.

In [ ]:
ORDER = ['p_ELO', 'p_xgb', V2, 'p_C3', 'p_C3raw', 'p_C3ens', 'p_C3ens_raw', 'b_C3_xgb', 'b_C3ens_raw_xgb', 'b_C3ens_xgb', 'b_C3_v2', 'b_C3ens_v2', 'b_3way', 'p_d35_full', 'b_C3ens_v35']
rows = []
for c in ORDER:
    p = pool[c].values
    m, lo, hi = brier_ci(p, yp, boots)
    r = {'model': LAB[c], 'Brier': np.mean((p - yp) ** 2), 'CI lo': lo, 'CI hi': hi}
    for ref, nm in (('p_C3', 'C3'), ('p_C3ens', 'C3ens')):
        if c != ref and c != 'p_ELO':
            d, dl, dh = paired_dbrier(p, pool[ref].values, yp, boots)
            r.update({f'd vs {nm}': d, f'lo vs {nm}': dl, f'hi vs {nm}': dh,
                      f'folds>{nm}': sum(np.mean((B[k][c] - B[k].Label) ** 2) < np.mean((B[k][ref] - B[k].Label) ** 2) for k in B)})
    rows.append(r)
MAIN = pd.DataFrame(rows)
print(MAIN.round(5).to_string(index=False))

The `folds>` columns count the folds (out of 4) in which the model has a lower Brier than the reference. Rows with TabPFN-3.5 are research-only and not production candidates.

In [ ]:
# Paired comparisons between the blends themselves
pairs = [('b_C3ens_xgb', 'b_C3ens_v2'), ('b_C3_xgb', 'b_C3_v2'), ('b_3way', 'b_C3ens_v2'), ('b_3way', 'b_C3ens_xgb'),
         ('b_C3ens_xgb', 'b_C3_xgb'), ('b_C3ens_xgb', 'b_C3ens_v35')]
for a, b_ in pairs:
    d, lo, hi = paired_dbrier(pool[a].values, pool[b_].values, yp, boots)
    w = sum(np.mean((B[k][a] - B[k].Label) ** 2) < np.mean((B[k][b_] - B[k].Label) ** 2) for k in B)
    print(f'{LAB[a]:30s} vs {LAB[b_]:42s} {d:+.5f} [{lo:+.5f}, {hi:+.5f}]  folds better {w}/4')

## Step 3b -- Brier by fold

In [ ]:
cols = ['p_ELO', 'p_xgb', V2, 'p_C3', 'p_C3ens', 'b_C3_xgb', 'b_C3ens_xgb', 'b_C3ens_v2', 'b_3way']
FT = pd.DataFrame([{'fold': k, **{LAB[c]: np.mean((B[k][c] - B[k].Label) ** 2) for c in cols}} for k in sorted(B)])
print(FT.round(5).to_string(index=False))
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(FT)); show = ['p_ELO', 'p_C3ens', 'p_xgb', 'b_C3ens_xgb', 'b_C3ens_v2']
col = ['#374151', '#ea580c', '#7c3aed', '#16a34a', '#2563eb']; w = 0.16
for i, (c, cc) in enumerate(zip(show, col)):
    ax.bar(x + (i - 2) * w, FT[LAB[c]], w, label=LAB[c], color=cc)
ax.set_xticks(x); ax.set_xticklabels([f'fold {k}' for k in FT['fold']])
ax.set_ylim(0.13, 0.19); ax.set_ylabel('Brier score (lower is better)'); ax.legend(ncol=2, fontsize=8)
ax.set_title('Walk-forward Brier by fold'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## Step 4 -- Correlation of residuals

Residual = outcome minus predicted probability, pooled over all rows. Lower correlation means more to gain from averaging.

In [ ]:
res = lambda c: yp - pool[c].values
for a, b_ in (('p_xgb', 'p_C3'), ('p_xgb', 'p_C3ens'), ('p_xgb', V2), ('p_C3', V2), ('p_C3ens', V2)):
    print(f'{LAB[a]:18s} vs {LAB[b_]:18s} {np.corrcoef(res(a), res(b_))[0, 1]:.4f}')

Residuals of the GRU and of either tabular model correlate at about 0.95-0.96, so they are far from identical, which is why a 50/50 average helps. XGBoost and TabPFN v2 residuals are more alike (about 0.98), consistent with the two blends being statistically indistinguishable.

## Step 5 -- Improvement over ELO

In [ ]:
be = np.mean((pool.p_ELO.values - yp) ** 2)
IMP = pd.DataFrame([{'model': LAB[c], 'Brier': np.mean((pool[c].values - yp) ** 2),
                     'improvement vs ELO (%)': 100 * (1 - np.mean((pool[c].values - yp) ** 2) / be)} for c in ORDER])
print(IMP.round(4).to_string(index=False))

## Step 6 -- Accuracy and AUC next to Brier (all models)

**Accuracy method.** Every match contributes two rows (one per player perspective), and accuracy is computed over **both perspectives**: a row is correct when `p > 0.5` agrees with the label. An exact tie (`p == 0.5`) counts as **half a point**. This is deliberate: TabPFN v2 outputs exactly 0.5 for about 683 rows per perspective, so any one-sided tie convention (always "win" or always "loss") would move its accuracy by up to +-2.4 pp, while half a point is neutral. The 95% CI is a bootstrap by match (2,000 resamples, seed 42, the same indices as every Brier CI above). AUC is pooled over all rows (no CI). `d acc` is the point difference to ELO in percentage points.

Accuracy is a coarse measure: it only looks at which side is above 0.5, so the top models lie within about 0.5 pp of each other and their CIs overlap. Brier and AUC are the measures that separate the models. Rows containing TabPFN-3.5 are research-only.

**Raw vs calibrated GRU.** The calibrated C3 (isotonic recalibration fitted on the validation block) is the row `C3 (calibrated)`; `C3 raw` is the same network without that step. Raw is slightly better pooled (Brier 0.1594 vs 0.1601), so since 15 November 2026 **production uses the raw C3** (no isotonic step). The rows `C3 raw`, `C3ens raw` (5 seeds) and `0.5 C3ens raw + 0.5 XGBoost B'` were added for this comparison; XGBoost B' stays calibrated in the blend (no raw output was saved for it).

In [ ]:
AC = summary_table(pool, ORDER, LAB, boots)
fmt = lambda r: (f"{r['Model']:46s} {r['Brier']:.5f} [{r['Brier lo']:.5f}, {r['Brier hi']:.5f}]  {r['% Brier vs ELO']:5.1f}%  "
                 f"{r['Acc %']:.2f} [{r['Acc lo']:.2f}, {r['Acc hi']:.2f}]  {r['dAcc pp']:+.2f}  {r['AUC']:.4f}  ties {r['ties']}")
print(f"{'Model':46s} {'Brier [95% CI]':32s} {'vs ELO':>6s}  Accuracy % [95% CI]    d acc pp  AUC")
for _, r in AC.iterrows():
    print(fmt(r))
print()
print('| Model | Brier [95% CI] | % Brier vs ELO | Accuracy [95% CI] | d acc vs ELO (pp) | AUC |')
print('|---|---|---|---|---|---|')
for _, r in AC.iterrows():
    print(f"| {r['Model']} | {r['Brier']:.5f} [{r['Brier lo']:.5f}, {r['Brier hi']:.5f}] | {r['% Brier vs ELO']:.1f}% | "
          f"{r['Acc %']:.2f}% [{r['Acc lo']:.2f}, {r['Acc hi']:.2f}] | {r['dAcc pp']:+.2f} | {r['AUC']:.4f} |")

Every GRU model and blend is about +1.3 to +1.9 pp more accurate than ELO (the tabular models XGBoost B' and TabPFN v2 about +0.6 to +0.8 pp), but the accuracy CIs of the GRU and the blends overlap; the ordering by Brier is the more reliable one.

## Step 7 -- Symmetry check

A match appears twice (player A vs B and B vs A), so a well-formed win probability satisfies `p_AB + p_BA = 1`. Below, `|p_AB + p_BA - 1|` per match for each model. ELO and TabPFN are exactly symmetric by construction (zero); the GRU models (C3, C3ens) and XGBoost are not, because the two perspectives are separate inputs. The last columns show the effect of the symmetrisation `p_sym = (p_AB + 1 - p_BA) / 2` on pooled Brier.

In [ ]:
SYM = ['p_ELO', V2, 'p_d35_full', 'p_C3', 'p_C3raw', 'p_C3ens', 'p_C3ens_raw', 'p_xgb', 'b_C3ens_raw_xgb']
rows = []
for c in SYM:
    dev = pair_dev(pool, c)
    ps = symmetrize(pool, c)
    b0, b1 = np.mean((pool[c].values - yp) ** 2), np.mean((ps - yp) ** 2)
    rows.append({'model': LAB[c], 'mean |dev|': dev.mean(), 'max |dev|': dev.max(), 'share > 0.01': (dev > 0.01).mean(),
                 'Brier': b0, 'Brier sym': b1, 'd Brier': b1 - b0})
print(pd.DataFrame(rows).round(5).to_string(index=False))

The asymmetry of the GRU models and XGBoost is real (up to 0.1-0.26 for single matches) but its effect on Brier is negligible: symmetrisation changes pooled Brier by at most about -0.0007 (XGBoost), and by -0.00003 to -0.0003 for the GRU rows and the blend, far inside the confidence intervals above. The models are therefore used as they are.

## Conclusion

- **A GRU + tabular blend is significantly better than the GRU on its own and than the 5-seed GRU ensemble.** Both 0.5 C3ens + 0.5 XGBoost and 0.5 C3ens + 0.5 TabPFN v2 have a paired dBrier vs C3ens whose CI lies below 0 (about -0.0011 and -0.0016); the XGBoost blend wins in 3 of 4 folds, the TabPFN v2 blend in 4 of 4.
- **C3ens + XGBoost is statistically indistinguishable from C3ens + TabPFN v2** (paired CI includes 0), and the 3-way blend adds nothing over the XGBoost blend (paired dBrier +0.00004, CI includes 0; its CI vs C3ens only just reaches 0).
- **XGBoost is cheap to serve** (a small tree ensemble, fast on CPU, no large pre-trained network), so it is the practical tabular partner for the GRU.
- **TabPFN-3.5 never goes to production** (non-commercial licence); its blend with the GRU is a research reference only.
- Differences between blends are in the third decimal of Brier, so they are small in absolute terms even when statistically clear.

**Limitations.**
- Weights are fixed 50/50 (and 1/3), not tuned; a tuned weight might be slightly better, but would need its own validation fold.
- C3ens is a 5-seed average whereas C3, XGBoost and TabPFN v2 are single-seed, so blends against C3 and against C3ens differ partly by ensembling.
- Test matches are shared across models; only paired CIs are used to compare.
- All inputs are isotonic-calibrated; raw model outputs were not compared.